# Đồ án môn CS419 – Truy xuất Thông tin
## Tập dữ liệu Cranfield

Notebook này trình bày toàn bộ pipeline truy xuất thông tin trên tập dữ liệu **Cranfield** (1 400 tài liệu, 225 câu truy vấn) sử dụng hai mô hình:
- **Vector Space Model (VSM)** với trọng số TF-IDF và độ đo Cosine Similarity
- **Okapi BM25**

In [1]:
import re, math, os, csv
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import word_tokenize
from num2words import num2words

for pkg in ['punkt', 'punkt_tab', 'stopwords']:
    nltk.download(pkg, quiet=True)

CRANFIELD = Path('Cranfield')
TEST      = Path('TEST')


## 1. Tải Dữ liệu (Data Loading)

Tập Cranfield gồm:
- **1 400** tài liệu khoa học về khí động học
- **225** câu truy vấn tiếng Anh
- **225** file đánh giá độ liên quan (relevance judgements)

In [2]:
def load_documents(folder):
    docs = {}
    for f in sorted(os.listdir(folder)):
        if f.endswith('.txt'):
            doc_id = int(f.split('.')[0])
            docs[doc_id] = open(os.path.join(folder, f), encoding='utf-8').read()
    return docs

def load_queries(query_file):
    queries = {}
    with open(query_file, encoding='utf-8') as f:
        for row in csv.reader(f, delimiter='\t'):
            if len(row) >= 2:
                queries[int(row[0])] = row[1]
    return queries

def load_relevance(result_path):
    rels = defaultdict(list)
    for fname in os.listdir(result_path):
        qid = int(fname.split('.')[0])
        df  = pd.read_csv(os.path.join(result_path, fname),
                          sep=r'\s+', header=None,
                          names=['QueryID','DocID','Rating'], engine='python')
        df  = df.dropna(subset=['DocID','Rating'])
        rels[qid] = [int(r.DocID) for r in df.itertuples() if int(r.Rating) != -1]
    return rels


In [3]:
documents  = load_documents(CRANFIELD)
queries    = load_queries(TEST / 'query.txt')
query_rels = load_relevance(TEST / 'RES')

print(f"Tài liệu : {len(documents)}")
print(f"Câu truy vấn : {len(queries)}")
print(f"Relevance files : {len(query_rels)}")


Tài liệu : 1400
Câu truy vấn : 225
Relevance files : 225


## 2. Tiền xử lý (Preprocessing)

Pipeline xử lý văn bản gồm 5 bước:

```
Văn bản thô
  → Lowercase
  → Mở rộng viết tắt (Regex)
  → Chuyển số thành chữ
  → Tokenize + Lọc Stopwords
  → Stemming (Snowball)
```

### 2.1. Làm sạch & Mở rộng viết tắt

In [4]:
ABBREVIATIONS = {
    r'\bfig\.?\b':    'figure',
    r'\bref\.?\b':    'reference',
    r'\bapprox\.?\b': 'approximately',
    r'\beq\.?\b':     'equation',
    r'\bsq\.?\b':     'square',
    r'\bno\.?\b':     'number',
    r'\be\.g\.?\b':  'for example',
    r'\bi\.e\.?\b':  'that is',
    r'\bsec\.?\b':    'section',
}

CUSTOM_STOPWORDS = {'ii','iii','iv','vi','vii','viii','ix','xi','xii'}

def replace_number(match):
    try:
        return num2words(float(match.group())).replace('-',' ').replace(',','')
    except Exception:
        return match.group()


### 2.2. Tokenization & Stopwords

In [5]:
def build_stopwords():
    sw = set(stopwords.words('english'))
    sw.update(CUSTOM_STOPWORDS)
    return sw

STOP_WORDS = build_stopwords()


#### Ví dụ chạy tay Pipeline Tiền Xử Lý (Query 1)

**Câu gốc:**
`"what similarity laws must be obeyed when constructing aeroelastic models of heated high speed aircraft ."`

**Bước 1: Chuyển chữ thường & Loại bỏ ký tự đặc biệt (Regex)**
`"what similarity laws must be obeyed when constructing aeroelastic models of heated high speed aircraft"`

**Bước 2: Tách từ (Tokenize)**
`['what', 'similarity', 'laws', 'must', 'be', 'obeyed', 'when', 'constructing', 'aeroelastic', 'models', 'of', 'heated', 'high', 'speed', 'aircraft']`

**Bước 3: Lọc Stopword & Từ ngắn (length <= 2)**
* Các từ bị loại: `what`, `must`, `be`, `when`, `of` (Stopword)
* Các từ giữ lại: `['similarity', 'laws', 'obeyed', 'constructing', 'aeroelastic', 'models', 'heated', 'high', 'speed', 'aircraft']`

**Bước 4: Stemming (Snowball)**
* `similarity` → `similar`
* `laws` → `law`
* `obeyed` → `obey`
* `constructing` → `construct`
* `aeroelastic` → `aeroelast`
* `models` → `model`
* `heated` → `heat`
* `high` → `high`
* `speed` → `speed`
* `aircraft` → `aircraft`

**Kết quả cuối cùng:**
`['similar', 'law', 'obey', 'construct', 'aeroelast', 'model', 'heat', 'high', 'speed', 'aircraft']`


### 2.3. Stemming

#### Thuật toán Snowball Stemmer

Snowball Stemmer áp dụng tập quy tắc **suffix-stripping** để đưa từ về dạng gốc. Ví dụ:

| Từ gốc | Sau Stemming |
|--------|-------------|
| *flowing* | flow |
| *studies* | studi |
| *aerodynamics* | aerodynam |
| *boundary* | boundari |

Mục đích: đồng nhất biến thể từ để tăng khả năng match giữa query và tài liệu.

In [6]:
STEMMER = SnowballStemmer('english')

examples = ['flowing', 'studies', 'aerodynamics', 'boundary', 'computed', 'material', 'photoelastic']
for w in examples:
    print(f"  {w:20s} → {STEMMER.stem(w)}")


  flowing              → flow
  studies              → studi
  aerodynamics         → aerodynam
  boundary             → boundari
  computed             → comput
  material             → materi
  photoelastic         → photoelast


### 2.4. Pipeline hoàn chỉnh

In [7]:
def process_document(document, stop_words=None, stemmer=None):
    if stop_words is None: stop_words = STOP_WORDS
    if stemmer    is None: stemmer    = STEMMER

    text = document.lower().replace('-', ' ')
    for pat, repl in ABBREVIATIONS.items():
        text = re.sub(pat, repl, text)

    text = re.sub(r'\b\d+(?:\.\d+)?\b', replace_number, text)
    text = re.sub(r'[^a-z0-9\s]', '', text)

    tokens = word_tokenize(text)
    tokens = [w for w in tokens
              if not (len(w) <= 2 and w.isalpha()) and w not in stop_words]
    return [stemmer.stem(t) for t in tokens]

def process_documents(doc_ids, documents):
    processed, all_tokens = {}, []
    for doc_id in doc_ids:
        tokens = process_document(documents[doc_id])
        processed[doc_id] = tokens
        all_tokens.extend(tokens)
    return processed, all_tokens


In [8]:
doc_ids = list(documents.keys())
processed_docs, all_tokens = process_documents(doc_ids, documents)


In [9]:
# Thống kê trước xử lý (áp dụng regex cơ bản để đếm token sạch)
def raw_tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

raw_tokens_all = []
for doc in documents.values():
    raw_tokens_all.extend(raw_tokenize(doc))

raw_vocab   = set(raw_tokens_all)
raw_avg_len = len(raw_tokens_all) / len(documents)

# Thống kê sau xử lý
proc_vocab   = set(all_tokens)
proc_avg_len = sum(len(t) for t in processed_docs.values()) / len(processed_docs)

before    = [len(raw_vocab),  round(raw_avg_len,  2)]
after     = [len(proc_vocab), round(proc_avg_len, 2)]
reduction = [f"{(b-a)/b*100:.2f}%" for b, a in zip(before, after)]

stats = pd.DataFrame({
    'Trước xử lý': before,
    'Sau xử lý'  : after,
    'Giảm'       : reduction,
}, index=['Term', 'Độ dài trung bình'])

stats.style\
    .set_caption("Thống kê Tiền xử lý")\
    .set_properties(**{'text-align': 'right'})\
    .set_properties(subset=pd.IndexSlice[:, ['Trước xử lý','Sau xử lý']], **{'font-weight': 'bold'})\
    .set_table_styles([
        {'selector': 'th', 'props': [('color','#3b3f8c'),('font-weight','bold'),('text-align','center')]},
        {'selector': 'td', 'props': [('padding','8px 16px')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f4f5ff')]},
        {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px')]},
    ])


,Trước xử lý,Sau xử lý,Giảm
Term,7472.000000,4452.000000,40.42%
Độ dài trung bình,161.910000,95.580000,40.97%


## 3. Xây dựng Từ điển & Chỉ mục

### 3.1. Vocabulary

In [10]:
def build_vocabulary(processed_docs):
    all_terms = set()
    for tokens in processed_docs.values():
        all_terms.update(tokens)
    vocab      = sorted(all_terms)
    vocab_index = {term: i for i, term in enumerate(vocab)}
    return vocab, vocab_index

vocab, vocab_index = build_vocabulary(processed_docs)
print(f"Vocabulary size: {len(vocab)}")


Vocabulary size: 4452


### 3.2. Inverted Index

#### Công thức Inverted Index

Cấu trúc lưu trữ:
```
term → {
  'nDoc'    : số tài liệu chứa term,
  'postings': [(doc_id, tfidf_weight), ...]
}
```
Giúp tra cứu nhanh O(1) thay vì quét toàn bộ ma trận.

In [11]:
def compute_tf(tokens, vocab_index):
    tf = np.zeros(len(vocab_index))
    for t in tokens:
        if t in vocab_index:
            tf[vocab_index[t]] += 1
    n = len(tokens)
    return tf / n if n > 0 else tf

def compute_global_df(processed_docs, vocab_index):
    df = np.zeros(len(vocab_index))
    for tokens in processed_docs.values():
        for t in set(tokens):
            if t in vocab_index:
                df[vocab_index[t]] += 1
    return df


## 4. Các Mô hình Truy xuất

### 4.1. Vector Space Model (VSM)

Mỗi tài liệu và câu truy vấn được biểu diễn dưới dạng **vector trọng số** trong không gian từ vựng. Tài liệu có vector cosine gần nhất với query sẽ được xếp hạng cao nhất.

#### Công thức TF

$$TF(t,d) = \frac{f_{t,d}}{|d|}$$

Trong đó $f_{t,d}$ là số lần term $t$ xuất hiện trong tài liệu $d$, $|d|$ là **tổng số token** trong $d$ (chuẩn hóa theo độ dài).

**Ví dụ:** doc có 95 token, term *'flow'* xuất hiện 3 lần → $TF = 3/95 \approx 0.0316$

#### Công thức IDF (VSM)

$$IDF(t) = \ln\left(\frac{N}{df_t}\right)$$

Trong đó $N = 1400$ (tổng số tài liệu), $df_t$ là số tài liệu chứa term $t$. Dùng logarithm tự nhiên $\ln$ (hàm `math.log` trong Python).

#### Công thức TF-IDF

$$w(t,d) = TF(t,d) \times IDF(t) = \frac{f_{t,d}}{|d|} \times \ln\frac{N}{df_t}$$

In [12]:
def compute_tfidf(processed_docs, vocab_index):
    N = len(processed_docs)
    global_df = compute_global_df(processed_docs, vocab_index)
    sorted_vocab = sorted(vocab_index, key=vocab_index.get)
    tfidf = np.zeros((N, len(vocab_index)))

    for doc_id, tokens in processed_docs.items():
        tf = compute_tf(tokens, vocab_index)
        for i in range(len(sorted_vocab)):
            if tf[i] > 0 and global_df[i] > 0:
                tfidf[doc_id - 1][i] = round(tf[i] * math.log(N / global_df[i]), 6)
    return tfidf

def cosine_similarity(v1, v2):
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    return np.dot(v1, v2) / (n1 * n2) if n1 and n2 else 0.0

def create_vsm_index(tfidf, processed_docs, vocab_index):
    df_dict = defaultdict(set)
    for doc_id, tokens in processed_docs.items():
        for t in set(tokens): df_dict[t].add(doc_id)

    index = {}
    for term, doc_ids in df_dict.items():
        idx = vocab_index.get(term)
        if idx is None: continue
        postings = [(d, tfidf[d-1][idx]) for d in doc_ids if tfidf[d-1][idx] > 0]
        index[term] = {'nDoc': len(doc_ids), 'postings': postings}
    return index

def search_vsm(query, tfidf, vsm_index, vocab_index, k=20):
    q_tokens = process_document(query)
    q_vec    = np.zeros(len(vocab_index))
    for t in q_tokens:
        if t in vocab_index: q_vec[vocab_index[t]] += 1
    n = len(q_tokens)
    if n > 0: q_vec /= n

    N = tfidf.shape[0]
    for t in set(q_tokens):
        if t in vsm_index:
            idx = vocab_index[t]
            q_vec[idx] *= math.log(N / vsm_index[t]['nDoc'])

    candidates = set()
    for t in q_tokens:
        if t in vsm_index:
            candidates.update(d for d, _ in vsm_index[t]['postings'])

    sims = [(d, cosine_similarity(q_vec, tfidf[d-1])) for d in candidates]
    return sorted(sims, key=lambda x: x[1], reverse=True)[:k]


In [13]:
tfidf_matrix = compute_tfidf(processed_docs, vocab_index)
vsm_index    = create_vsm_index(tfidf_matrix, processed_docs, vocab_index)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")


TF-IDF matrix shape: (1400, 4452)


### Xuất thống kê toàn bộ từ vựng ra `tfidf_terms.csv`

Tính toán tổng tần suất (Total TF), số tài liệu chứa từ (DF), IDF và TF-IDF lớn nhất cho từng từ trong bộ dữ liệu.

In [14]:
term_stats = []
N = len(processed_docs)

# Đếm tổng tần suất (Total TF) cho toàn bộ từ vựng
total_tf_counter = Counter()
for tokens in processed_docs.values():
    total_tf_counter.update(tokens)

for term, data in vsm_index.items():
    df = data['nDoc']
    idf = math.log(N / df)
    idx = vocab_index[term]
    max_tfidf = np.max(tfidf_matrix[:, idx])
    
    term_stats.append({
        'Term': term,
        'Total_TF': total_tf_counter[term],
        'DF': df,
        'IDF': round(idf, 4),
        'Max_TFIDF': round(max_tfidf, 4)
    })

df_terms = pd.DataFrame(term_stats).sort_values(by='Total_TF', ascending=False)
df_terms.to_csv('tfidf_terms.csv', index=False)

print(f"Đã lưu {len(df_terms)} từ vựng ra file 'tfidf_terms.csv'")
display(df_terms.head(10))


Đã lưu 4452 từ vựng ra file 'tfidf_terms.csv'


,Term,Total_TF,DF,IDF,Max_TFIDF
2,flow,2082,730,0.6512,0.1108
67,number,1500,633,0.7938,0.0803
69,pressur,1391,552,0.9307,0.1241
44,boundari,1216,470,1.0915,0.1213
37,layer,1164,414,1.2184,0.1354
50,result,1088,692,0.7046,0.0587
75,two,1071,602,0.8440,0.0675
192,point,1065,476,1.0788,0.1425
144,one,1032,561,0.9145,0.1006
43,effect,996,540,0.9527,0.0697


#### Triển khai VSM

In [15]:
# Dùng Query 1 theo yêu cầu để minh họa
demo_query = queries[1]
print(f"Query 1: {demo_query}")
print("\nTop 5 kết quả (VSM):")
for doc_id, score in search_vsm(demo_query, tfidf_matrix, vsm_index, vocab_index, k=5):
    print(f"  Doc {doc_id:4d}  score={score:.4f}")


Query 1: what similarity laws must be obeyed when constructing aeroelastic models of heated high speed aircraft .

Top 5 kết quả (VSM):
  Doc   51  score=0.2780
  Doc  184  score=0.2437
  Doc   12  score=0.2185
  Doc  359  score=0.1956
  Doc  746  score=0.1916


### Xuất thống kê chi tiết VSM ra CSV

Tạo file `query_vsm.csv` phân rã điểm số Cosine Similarity thành các thành phần nhỏ (TF-IDF Query, TF-IDF Doc, Dot Product) cho từng term của Top 10 tài liệu đối với toàn bộ 225 câu truy vấn.

In [16]:
query_vsm_data = []

for qid, qtext in queries.items():
    q_tokens = process_document(qtext)
    
    # Dựng vector query
    q_vec = np.zeros(len(vocab_index))
    for t in q_tokens:
        if t in vocab_index: q_vec[vocab_index[t]] += 1
    if len(q_tokens) > 0: q_vec /= len(q_tokens)
    
    N = tfidf_matrix.shape[0]
    for t in set(q_tokens):
        if t in vsm_index:
            q_vec[vocab_index[t]] *= math.log(N / vsm_index[t]['nDoc'])
            
    q_norm = np.linalg.norm(q_vec)
    top10_docs = search_vsm(qtext, tfidf_matrix, vsm_index, vocab_index, k=10)
    
    for rank, (doc_id, total_sim) in enumerate(top10_docs, 1):
        d_vec = tfidf_matrix[doc_id-1]
        d_norm = np.linalg.norm(d_vec)
        
        for t in set(q_tokens):
            if t in vocab_index:
                idx = vocab_index[t]
                w_q = q_vec[idx]
                w_d = d_vec[idx]
                if w_q > 0 and w_d > 0:
                    dot_contrib = w_q * w_d
                    query_vsm_data.append({
                        'Query_ID': qid,
                        'Doc_ID': doc_id,
                        'Rank': rank,
                        'Term': t,
                        'W_query': round(w_q, 4),
                        'W_doc': round(w_d, 4),
                        'Dot_Product_Contrib': round(dot_contrib, 4),
                        'Query_Norm': round(q_norm, 4),
                        'Doc_Norm': round(d_norm, 4),
                        'Total_Cosine_Sim': round(total_sim, 4)
                    })

df_query_vsm = pd.DataFrame(query_vsm_data)
df_query_vsm.to_csv('query_vsm.csv', index=False)
print(f"Đã lưu {len(df_query_vsm)} dòng vào query_vsm.csv")
display(df_query_vsm.head(10))


Đã lưu 9782 dòng vào query_vsm.csv


,Query_ID,Doc_ID,Rank,Term,W_query,W_doc,Dot_Product_Contrib,Query_Norm,Doc_Norm,Total_Cosine_Sim
0,1,51,1,aircraft,0.2710,0.2556,0.0693,0.9577,0.5059,0.2780
1,1,51,1,model,0.1880,0.0788,0.0148,0.9577,0.5059,0.2780
2,1,51,1,speed,0.1425,0.0149,0.0021,0.9577,0.5059,0.2780
3,1,51,1,heat,0.1382,0.1014,0.0140,0.9577,0.5059,0.2780
4,1,51,1,construct,0.3210,0.0673,0.0216,0.9577,0.5059,0.2780
5,1,51,1,similar,0.2024,0.0636,0.0129,0.9577,0.5059,0.2780
6,1,184,2,aeroelast,0.3958,0.1574,0.0623,0.9577,0.4387,0.2437
7,1,184,2,aircraft,0.2710,0.0359,0.0097,0.9577,0.4387,0.2437
8,1,184,2,model,0.1880,0.0748,0.0141,0.9577,0.4387,0.2437
9,1,184,2,similar,0.2024,0.0805,0.0163,0.9577,0.4387,0.2437


#### Ví dụ tính tay TF-IDF (1 term cụ thể)

**Ví dụ: "aeroelast" trong tài liệu số 51**
+ N: tổng số tài liệu (=1400)
+ \|d\|: độ dài tài liệu số 51 (=162)
+ DF: số tài liệu chứa từ "aeroelast" (=113)
+ TF(f): số lần "aeroelast" xuất hiện trong tài liệu 51 (=5)

| TERM | TF(f) | \|d\| | DF | IDF | W |
|:---:|:---:|:---:|:---:|:---:|:---:|
| aeroelast | 5 | 162 | 113 | 2.517 | 0.0777 |

$$idf_{aeroelast} = \ln\left(\frac{1400}{113}\right) \approx 2.517$$

$$w_{aeroelast} = \frac{5}{162} \times 2.517 \approx 0.0777$$


### 4.2. Okapi BM25

BM25 là mô hình xác suất, bổ sung **chuẩn hóa độ dài tài liệu** ($b$) và **bão hòa tần suất** ($k_1$) so với TF-IDF thông thường.

#### Công thức IDF (BM25)

$$IDF_{BM25}(t) = \ln\left(\frac{N - df_t + 0.5}{df_t + 0.5} + 1\right)$$

Khác với VSM: BM25 dùng công thức IDF có tham số smoothing $+1$ bên trong, tránh IDF âm khi $df_t > N/2$.

#### Công thức Score BM25

$$\text{score}(d,q) = \sum_{t \in q} IDF_{BM25}(t) \cdot \frac{f_{t,d} \cdot (k_1+1)}{f_{t,d} + k_1\left(1 - b + b \cdot \dfrac{|d|}{avgdl}\right)}$$

Tham số tối ưu (qua Grid Search): $k_1 = 2.0$, $b = 0.6$, $avgdl \approx 91$ token/tài liệu

### 4.2.1. Cấu trúc Chỉ mục đảo ngược & Thuật toán lập chỉ mục cho Okapi BM25

Để tối ưu hóa hiệu năng, mô hình BM25 đã được cải tiến để sử dụng cấu trúc **Chỉ mục đảo ngược (Inverted Index)** thay thế cho việc quét tuyến tính qua toàn bộ tài liệu.

#### 1. Cấu trúc dữ liệu chỉ mục
* **Chỉ mục đảo ngược (`self.index`)**:
  Một cấu trúc kiểu Từ điển (`dict`) trong đó khóa là từ khóa (`term`) và giá trị tương ứng là một danh sách chứa các cặp `(doc_id, tf)` biểu thị tần suất xuất hiện của từ khóa đó trong tài liệu tương ứng (gọi là **Postings List**):
  ```python
  term -> [(doc_id_1, tf_1), (doc_id_2, tf_2), ...]
  ```
* **Độ dài tài liệu (`self.doc_len`)**:
  Để phục vụ cho phần mẫu số trong công thức BM25, chúng ta duy trì một bảng độ dài tài liệu toàn cục dạng `{doc_id: length}`.

#### 2. Thuật toán lập chỉ mục (Indexing Algorithm)
1. Duyệt qua từng tài liệu trong cơ sở dữ liệu.
2. Đếm số lần xuất hiện (tần suất từ khóa) bằng `Counter`.
3. Thêm các thông tin `(doc_id, tf)` vào danh sách postings của từ khóa đó trong `self.index`.
4. Tính độ dài trung bình (`avgdl`) của toàn bộ tập dữ liệu.
5. Tính sẵn IDF của từng từ khóa dựa trên độ dài postings list của từ đó (chính là $df$ - Document Frequency).

#### 3. Thuật toán tìm kiếm (Retrieval Algorithm)
1. Nhận danh sách các từ khóa từ câu truy vấn (`query_tokens`).
2. Với mỗi từ khóa truy vấn, tra cứu postings list của nó trong chỉ mục đảo ngược.
3. Chỉ thực hiện tính toán điểm số BM25 cho các tài liệu xuất hiện trong postings list tương ứng.
4. Tích lũy điểm vào bảng tổng điểm `scores[doc_id]` và trả về Top K tài liệu cao nhất.

*Độ phức tạp thời gian giảm từ quét tuyến tính $O(N \cdot |Q|)$ xuống chỉ còn $O(|Q| \cdot L)$ với $L$ là độ dài trung bình của postings list ($L \ll N$).*

In [17]:
class BM25_Legacy:
    def __init__(self, processed_docs, k1=2.0, b=0.6):
        self.k1, self.b = k1, b
        self.N   = len(processed_docs)
        self.doc_len   = {}
        self.doc_freqs = {}
        self.nd        = {}
        self.idf       = {}
        self.avgdl     = 0
        self._initialize(processed_docs)

    def _initialize(self, docs):
        total = 0
        for doc_id, tokens in docs.items():
            self.doc_len[doc_id] = len(tokens)
            total += len(tokens)
            freq = Counter(tokens)
            self.doc_freqs[doc_id] = freq
            for w in freq: self.nd[w] = self.nd.get(w, 0) + 1

        self.avgdl = total / self.N if self.N > 0 else 0
        for w, df in self.nd.items():
            self.idf[w] = math.log((self.N - df + 0.5) / (df + 0.5) + 1.0)

    def get_scores(self, query_tokens, k=20):
        scores = {}
        for doc_id, freq in self.doc_freqs.items():
            s, dl = 0.0, self.doc_len[doc_id]
            for t in query_tokens:
                if t not in freq: continue
                f = freq[t]
                num = f * (self.k1 + 1)
                den = f + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                s  += self.idf.get(t, 0) * (num / den)
            if s > 0: scores[doc_id] = s
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]

class BM25:
    def __init__(self, processed_docs, k1=2.0, b=0.6):
        self.k1, self.b = k1, b
        self.N   = len(processed_docs)
        self.doc_len   = {}
        self.index     = defaultdict(list) # Inverted Index: term -> [(doc_id, tf), ...]
        self.idf       = {}
        self.avgdl     = 0
        self._initialize(processed_docs)

    def _initialize(self, docs):
        total = 0
        for doc_id, tokens in docs.items():
            self.doc_len[doc_id] = len(tokens)
            total += len(tokens)
            freq = Counter(tokens)
            for w, f in freq.items():
                self.index[w].append((doc_id, f))

        self.avgdl = total / self.N if self.N > 0 else 0
        for w, postings in self.index.items():
            df = len(postings)
            self.idf[w] = math.log((self.N - df + 0.5) / (df + 0.5) + 1.0)

    def get_scores(self, query_tokens, k=20):
        scores = defaultdict(float)
        for t in query_tokens:
            if t not in self.index: continue
            idf_t = self.idf.get(t, 0.0)
            if idf_t <= 0: continue
            for doc_id, f in self.index[t]:
                dl = self.doc_len[doc_id]
                num = f * (self.k1 + 1)
                den = f + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                scores[doc_id] += idf_t * (num / den)
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]

def search_bm25(query, bm25_model, k=20):
    tokens = process_document(query)
    return bm25_model.get_scores(tokens, k=k)


In [18]:
bm25_model = BM25(processed_docs)
print(f"avgdl = {bm25_model.avgdl:.1f} tokens  |  N = {bm25_model.N}")


avgdl = 95.6 tokens  |  N = 1400


#### Demo BM25 – Query 1

In [19]:
print(f"Query 1: '{queries[1]}'")
print("\nTop 5 kết quả (BM25):")
for doc_id, score in search_bm25(queries[1], bm25_model, k=5):
    print(f"  Doc {doc_id:4d}  BM25 = {score:.4f}")


Query 1: 'what similarity laws must be obeyed when constructing aeroelastic models of heated high speed aircraft .'

Top 5 kết quả (BM25):
  Doc   51  BM25 = 25.3230
  Doc  486  BM25 = 22.3925
  Doc   12  BM25 = 20.3612
  Doc  184  BM25 = 19.1581
  Doc  878  BM25 = 17.3155


### Xuất thống kê chi tiết BM25 ra CSV

Tạo hai file CSV chứa các thành phần tính toán chi tiết của BM25:
- `term_bm25.csv`: Bảng thành phần (DF, IDF) và Max Score của tất cả các từ trong bộ dữ liệu.
- `query_bm25.csv`: Bảng phân tách điểm số BM25 chi tiết đến từng term cho Top 10 tài liệu được truy hồi của toàn bộ 225 câu truy vấn.

In [20]:
# 1. Xuất term_bm25.csv
term_bm25_data = []
for term, idf in bm25_model.idf.items():
    df = len(bm25_model.index.get(term, []))
    
    max_tf = 0
    max_score = 0.0
    
    if term in vsm_index:
        for doc_id, _ in vsm_index[term]['postings']:
            dl = bm25_model.doc_len[doc_id]
            # Lấy tf (f) trực tiếp từ Inverted Index của BM25
            f = 0
            if term in bm25_model.index:
                for d, freq in bm25_model.index[term]:
                    if d == doc_id:
                        f = freq
                        break
            if f > max_tf: max_tf = f
            
            num = f * (bm25_model.k1 + 1)
            den = f + bm25_model.k1 * (1 - bm25_model.b + bm25_model.b * dl / bm25_model.avgdl)
            score = idf * (num / den)
            if score > max_score: max_score = score

    term_bm25_data.append({
        'Term': term,
        'DF': df,
        'IDF_BM25': round(idf, 4),
        'Max_TF': max_tf,
        'Max_BM25_Score': round(max_score, 4)
    })

df_term_bm25 = pd.DataFrame(term_bm25_data).sort_values(by='IDF_BM25', ascending=False)
df_term_bm25.to_csv('term_bm25.csv', index=False)
print(f"Đã lưu {len(df_term_bm25)} dòng vào term_bm25.csv")


# 2. Xuất query_bm25.csv (Phân rã thành phần điểm cho Top 10 doc của mỗi Query)
query_bm25_data = []

for qid, qtext in queries.items():
    q_tokens = process_document(qtext)
    top10_docs = search_bm25(qtext, bm25_model, k=10)
    
    for rank, (doc_id, total_score) in enumerate(top10_docs, 1):
        dl = bm25_model.doc_len[doc_id]
        
        for t in q_tokens:
            # Lấy tf (f) trực tiếp từ Inverted Index của BM25
            f = 0
            if t in bm25_model.index:
                for d, freq in bm25_model.index[t]:
                    if d == doc_id:
                        f = freq
                        break
            idf = bm25_model.idf.get(t, 0.0)
            
            if f > 0 and idf > 0:
                num = f * (bm25_model.k1 + 1)
                den = f + bm25_model.k1 * (1 - bm25_model.b + bm25_model.b * dl / bm25_model.avgdl)
                term_score = idf * (num / den)
                
                query_bm25_data.append({
                    'Query_ID': qid,
                    'Doc_ID': doc_id,
                    'Rank': rank,
                    'Doc_Length': dl,
                    'Term': t,
                    'TF(f)': f,
                    'IDF_BM25': round(idf, 4),
                    'Numerator': round(num, 4),
                    'Denominator': round(den, 4),
                    'Term_BM25_Score': round(term_score, 4)
                })

df_query_bm25 = pd.DataFrame(query_bm25_data)
df_query_bm25.to_csv('query_bm25.csv', index=False)
print(f"Đã lưu {len(df_query_bm25)} dòng vào query_bm25.csv")
display(df_query_bm25.head(10))


Đã lưu 4452 dòng vào term_bm25.csv
Đã lưu 12217 dòng vào query_bm25.csv


,Query_ID,Doc_ID,Rank,Doc_Length,Term,TF(f),IDF_BM25,Numerator,Denominator,Term_BM25_Score
0,1,51,1,105,similar,3,2.2244,9.0,5.1182,3.9113
1,1,51,1,105,construct,2,3.5192,6.0,4.1182,5.1273
2,1,51,1,105,model,4,2.0660,12.0,6.1182,4.0521
3,1,51,1,105,heat,7,1.5197,21.0,9.1182,3.5000
4,1,51,1,105,speed,1,1.5665,3.0,3.1182,1.5071
5,1,51,1,105,aircraft,9,2.9752,27.0,11.1182,7.2252
6,1,486,2,139,similar,4,2.2244,12.0,6.5451,4.0782
7,1,486,2,139,law,3,3.2653,9.0,5.5451,5.2997
8,1,486,2,139,aeroelast,1,4.3272,3.0,3.5451,3.6618
9,1,486,2,139,model,5,2.0660,15.0,7.5451,4.1072


#### Ví dụ tính tay BM25 (1 term cụ thể)

**Ví dụ: "aeroelast" trong tài liệu số 51 (BM25)**
+ N: tổng số tài liệu (=1400)
+ DF: số tài liệu chứa từ "aeroelast" (=113)
+ TF(f): số lần "aeroelast" xuất hiện trong tài liệu 51 (=5)
+ k1: tham số bão hòa tần suất (=2.0)
+ b: tham số chuẩn hóa độ dài (=0.6)
+ \|d\|: độ dài tài liệu 51 (=162)
+ avgdl: độ dài trung bình của tài liệu (=91.22)

| TERM | TF(f) | DF | IDF | TF_comp | SCORE |
|:---:|:---:|:---:|:---:|:---:|:---:|
| aeroelast | 5 | 113 | 2.515 | 1.831 | 4.606 |

$$idf_{aeroelast} = \ln\left(\frac{1400 - 113 + 0.5}{113 + 0.5} + 1\right) \approx 2.515$$

$$score_{aeroelast} = 2.515 \times \frac{5 \times (2.0 + 1)}{5 + 2.0 \times \left(1 - 0.6 + 0.6 \times \frac{162}{91.22}\right)} = 2.515 \times \frac{15.000}{8.192} \approx 4.606$$


## 5. Đánh giá (Evaluation)

### Công thức MAP, P@k, Recall@k

$$MAP = \frac{1}{|Q|} \sum_{q=1}^{|Q|} AP(q) \qquad AP(q) = \frac{1}{R_q} \sum_{k=1}^{n} P(k) \cdot rel(k)$$

$$P@k = \frac{|\text{relevant} \cap \text{top-k}|}{k} \qquad Recall@k = \frac{|\text{relevant} \cap \text{top-k}|}{|\text{relevant}|}$$

Trong đó $R_q$ là tổng số tài liệu liên quan cho query $q$, $rel(k)=1$ nếu tài liệu tại vị trí $k$ là relevant.

### Triển khai

In [21]:
def average_precision(retrieved, relevant):
    relevant_set = set(relevant)
    if not relevant_set: return 0.0
    score, hits = 0.0, 0
    for i, doc in enumerate(retrieved, 1):
        if doc in relevant_set:
            hits  += 1
            score += hits / i
    return score / len(relevant_set)

def evaluate_model(results, query_rels, k=20):
    aps, ps, rs = [], [], []
    for qid, retrieved in results.items():
        if qid not in query_rels: continue
        relevant = set(query_rels[qid])
        if not relevant: continue
        top_k = retrieved[:k]
        hits  = sum(1 for d in top_k if d in relevant)
        ps.append(hits / k)
        rs.append(hits / len(relevant))
        aps.append(average_precision(retrieved, query_rels[qid]))
    return {
        'MAP':        round(float(np.mean(aps)), 4) if aps else 0.0,
        f'P@{k}':     round(float(np.mean(ps)),  4) if ps  else 0.0,
        f'Recall@{k}':round(float(np.mean(rs)),  4) if rs  else 0.0,
    }


#### Ví dụ chạy tay Đánh giá (Evaluation) cho Query 1 với k=5

**Giả sử Query 1 chạy qua mô hình VSM trả về top 5 kết quả:**
- **Relevant (Tập tài liệu thực sự liên quan)**: $R_{q_1} = 28$ tài liệu (bao gồm `51, 184, 12, ...`)
- **Retrieved (Top 5 tài liệu trả về)**: `[51, 184, 12, 359, 746]`
- **Intersection (Tài liệu truy xuất đúng trong top 5)**: `[51, 184, 12]` (3 tài liệu)

**1. Tính Precision@5 (P@5):**
Tỷ lệ tài liệu trả về đúng trên tổng số 5 tài liệu được hệ thống trả về.
$$P@5 = \frac{|\text{Relevant} \cap \text{Retrieved@5}|}{5} = \frac{3}{5} = 0.6000$$

**2. Tính Recall@5:**
Tỷ lệ tài liệu trả về đúng trên tổng số tài liệu thực sự liên quan của toàn hệ thống ($R_{q_1}=28$).
$$Recall@5 = \frac{|\text{Relevant} \cap \text{Retrieved@5}|}{|Relevant|} = \frac{3}{28} \approx 0.1071$$

**3. Tính Average Precision (AP):**
Xét từng vị trí $i$ từ 1 đến 5 để tính trung bình các Precision tại các vị trí dự đoán đúng (hits):
- Tại $i=1$, Doc `51` (Relevant ✅) $\rightarrow$ hits = 1, $P(1) = 1/1 = 1.000$
- Tại $i=2$, Doc `184` (Relevant ✅) $\rightarrow$ hits = 2, $P(2) = 2/2 = 1.000$
- Tại $i=3$, Doc `12` (Relevant ✅) $\rightarrow$ hits = 3, $P(3) = 3/3 = 1.000$
- Tại $i=4$, Doc `359` (Irrelevant ❌) $\rightarrow$ hits = 3, (Không cộng vào tổng vì $rel=0$)
- Tại $i=5$, Doc `746` (Irrelevant ❌) $\rightarrow$ hits = 3, (Không cộng vào tổng vì $rel=0$)

Tổng điểm = $1.000 + 1.000 + 1.000 = 3.000$
$$AP(q_1) = \frac{\text{Tổng điểm}}{|Relevant|} = \frac{3.000}{28} \approx 0.1071$$


In [72]:

# Chạy VSM – lấy Top 20 để đánh giá P@20, Recall@20
vsm_results = {}
for qid, qtext in queries.items():
    vsm_results[qid] = [d for d,_ in search_vsm(qtext, tfidf_matrix, vsm_index, vocab_index, k=1400)]

# Chạy BM25 – lấy Top 20 để đánh giá P@20, Recall@20
bm25_results = {}
for qid, qtext in queries.items():
    bm25_results[qid] = [d for d,_ in search_bm25(qtext, bm25_model, k=1400)]

vsm_metrics  = evaluate_model(vsm_results,  query_rels, k=20)
bm25_metrics = evaluate_model(bm25_results, query_rels, k=20)

summary = pd.DataFrame([vsm_metrics, bm25_metrics], index=['VSM', 'BM25'])
summary.style.set_caption("Kết quả Đánh giá – Cranfield 1400 docs / 225 queries").set_table_styles([
    {'selector':'th','props':[('color','#3b3f8c'),('font-weight','bold'),('text-align','center')]},
    {'selector':'td','props':[('padding','8px 18px'),('text-align','right'),('font-size','15px')]},
    {'selector':'tr:nth-child(even)','props':[('background-color','#f4f5ff')]},
])


,MAP,P@20,Recall@20
VSM,0.292300,0.157300,0.504800
BM25,0.311800,0.162200,0.518200


### Xuất Đánh giá chi tiết từng Query ra CSV

Tạo file `evaluation_per_query.csv` ghi nhận lại điểm AP, P@20, và Recall@20 của VSM và BM25 cho tất cả các câu truy vấn để dễ dàng phân tích sâu.

In [23]:
k = 20
eval_data = []

for qid in queries.keys():
    if qid not in query_rels: continue
    relevant = set(query_rels[qid])
    if not relevant: continue
    
    # Hàm tính metric nội bộ
    def calc_metrics(retrieved):
        top_k = retrieved[:k]
        hits = sum(1 for d in top_k if d in relevant)
        p_k = hits / k
        r_k = hits / len(relevant)
        ap = average_precision(retrieved, query_rels[qid])
        return ap, p_k, r_k
        
    ap_vsm, p_vsm, r_vsm = calc_metrics(vsm_results.get(qid, []))
    ap_bm25, p_bm25, r_bm25 = calc_metrics(bm25_results.get(qid, []))
    
    eval_data.append({
        'Query_ID': qid,
        'Num_Relevant': len(relevant),
        'AP_VSM': round(ap_vsm, 4),
        'P@20_VSM': round(p_vsm, 4),
        'Recall@20_VSM': round(r_vsm, 4),
        'AP_BM25': round(ap_bm25, 4),
        'P@20_BM25': round(p_bm25, 4),
        'Recall@20_BM25': round(r_bm25, 4)
    })

df_eval = pd.DataFrame(eval_data)
df_eval.to_csv('evaluation_per_query.csv', index=False)
print(f"Đã lưu {len(df_eval)} dòng vào evaluation_per_query.csv")
display(df_eval.head(10))


Đã lưu 225 dòng vào evaluation_per_query.csv


,Query_ID,Num_Relevant,AP_VSM,P@20_VSM,Recall@20_VSM,AP_BM25,P@20_BM25,Recall@20_BM25
0,1,28,0.2398,0.35,0.2500,0.1990,0.30,0.2143
1,2,24,0.2277,0.30,0.2500,0.2043,0.25,0.2083
2,3,8,0.6341,0.35,0.8750,0.6477,0.35,0.8750
3,4,2,0.5526,0.10,1.0000,0.5345,0.05,0.5000
4,5,4,0.2001,0.15,0.7500,0.3048,0.15,0.7500
5,6,4,0.1359,0.10,0.5000,0.1993,0.10,0.5000
6,7,5,0.1021,0.10,0.4000,0.1367,0.10,0.4000
7,8,11,0.1613,0.20,0.3636,0.1536,0.05,0.0909
8,9,3,1.0000,0.15,1.0000,0.5889,0.15,1.0000
9,10,8,0.1610,0.15,0.3750,0.1442,0.15,0.3750


## 6. Phân tích Best / Worst Case (BM25)

Phân tích những câu truy vấn mà mô hình trả về kết quả **tốt nhất** và **tệ nhất** để hiểu điểm mạnh và điểm yếu của hệ thống.

In [73]:
# Tính AP cho từng query
ap_per_query = {qid: average_precision(bm25_results[qid], query_rels.get(qid,[]))
                for qid in bm25_results if qid in query_rels}

best_qid  = max(ap_per_query, key=ap_per_query.get)
worst_qid = min(ap_per_query, key=ap_per_query.get)

print(f"Best  query: Q{best_qid}  AP={ap_per_query[best_qid]:.4f}")
print(f"Worst query: Q{worst_qid}  AP={ap_per_query[worst_qid]:.4f}")


Best  query: Q119  AP=1.0000
Worst query: Q13  AP=0.0000


### Best Case – BM25

In [74]:
qid = best_qid
print(f"Query #{qid}: {queries[qid]}")
print(f"AP = {ap_per_query[qid]:.4f}  |  Relevant docs: {len(query_rels.get(qid,[]))}")

top_doc = bm25_results[qid][0]
q_tokens_best = process_document(queries[qid])
dl = bm25_model.doc_len[top_doc]

# Bảng phân tích score từng term
rows = []
for t in q_tokens_best:
    # Lấy tf (f) trực tiếp từ Inverted Index của BM25
    f = 0
    if t in bm25_model.index:
        for d_id, freq in bm25_model.index[t]:
            if d_id == top_doc:
                f = freq
                break
    idf  = bm25_model.idf.get(t, 0)
    num  = f * (bm25_model.k1 + 1)
    den  = f + bm25_model.k1 * (1 - bm25_model.b + bm25_model.b * dl / bm25_model.avgdl)
    rows.append({'Term (query)': t, 'f(t,d)': f,
                 'IDF_BM25': round(idf,4),
                 'Numerator': round(num,4), 'Denominator': round(den,4),
                 'Term score': round(idf*(num/den) if den>0 and f>0 else 0, 4)})

sc_df = pd.DataFrame(rows)
total = sc_df['Term score'].sum()
print(f"\nDoc {top_doc} | |d|={dl} | Final BM25 score = {total:.4f}")
sc_df.style.set_caption(f"Score breakdown – Query {qid} × Doc {top_doc}").set_table_styles([
    {'selector':'th','props':[('color','#3b3f8c'),('font-weight','bold'),('text-align','center')]},
    {'selector':'td','props':[('padding','5px 12px'),('text-align','right')]},
    {'selector':'td:first-child','props':[('text-align','left'),('color','#3b3f8c')]},
])


Query #119: what is the effect of initial axisymmetric deviations from circularity on the non linear (large-deflection) load-deflection response of cylinders under hydrostatic pressure .
AP = 1.0000  |  Relevant docs: 1

Doc 926 | |d|=80 | Final BM25 score = 30.0282


,Term (query),"f(t,d)",IDF_BM25,Numerator,Denominator,Term score
0,effect,0,0.952400,0.000000,1.804400,0.000000
1,initi,1,2.549000,3.000000,2.804400,2.726800
2,axisymmetr,0,3.265300,0.000000,1.804400,0.000000
3,deviat,0,3.967800,0.000000,1.804400,0.000000
4,circular,2,2.306900,6.000000,3.804400,3.638200
5,non,0,3.093900,0.000000,1.804400,0.000000
6,linear,0,2.148100,0.000000,1.804400,0.000000
7,larg,0,1.890700,0.000000,1.804400,0.000000
8,deflect,3,2.773300,9.000000,4.804400,5.195200
9,load,6,1.944100,18.000000,7.804400,4.483900


### Worst Case – Query tìm kiếm kém chính xác nhất

In [26]:
qid = worst_qid
print(f"Query #{qid}: {queries[qid]}")
print(f"AP = {ap_per_query[qid]:.4f}\n")

relevant_docs = query_rels.get(qid, [])
retrieved_top5 = bm25_results[qid][:5]

print(f"Relevant docs (first 10): {relevant_docs[:10]}")
print(f"Retrieved top-5          : {retrieved_top5}")
overlap = set(retrieved_top5) & set(relevant_docs)
print(f"Overlap in top-5         : {overlap if overlap else 'Không có (False Positive hoàn toàn)'}")


Query #13: what is the basic mechanism of the transonic aileron buzz .
AP = 0.0000

Relevant docs (first 10): [64, 265, 65, 311]
Retrieved top-5          : [496, 903, 520, 643, 199]
Overlap in top-5         : Không có (False Positive hoàn toàn)


# Mở rộng: CLUSTERING

## Thuật toán chính

In [75]:
# ============================================================
# Section 6: Cluster-based Reranking
# BM25/VSM + Cluster-aware Score Boosting
# ============================================================

# ============================================================
# TỔNG QUAN HOẠT ĐỘNG CỦA ĐOẠN CODE
# ============================================================
# Mục tiêu của đoạn code này:
#   - Ban đầu ta đã có các mô hình truy xuất tài liệu như BM25 hoặc VSM.
#   - BM25/VSM trả về danh sách tài liệu theo độ liên quan với query.
#   - Tuy nhiên, ranking ban đầu chỉ dựa trên điểm retrieval trực tiếp.
#   - Đoạn code này thêm một tín hiệu mới: thông tin cụm/chủ đề của tài liệu.
#   - Ý tưởng là: nếu nhiều tài liệu top đầu cùng nằm trong một cluster, cluster đó có
#     khả năng liên quan mạnh đến query, nên các tài liệu thuộc cluster đó sẽ được boost nhẹ.
#
# Luồng xử lý chính:
#   1. Lấy ma trận TF-IDF của toàn bộ document.
#      - Mỗi document đang được biểu diễn bằng vector TF-IDF rất nhiều chiều.
#      - Số chiều thường bằng số lượng từ vựng trong corpus.
#
#   2. Dùng TruncatedSVD để giảm chiều TF-IDF.
#      - TF-IDF gốc thường rất sparse và nhiều chiều.
#      - KMeans chạy trực tiếp trên ma trận quá nhiều chiều có thể chậm và nhiễu.
#      - SVD giúp nén document vector xuống 100 chiều.
#      - Sau SVD, mỗi document được biểu diễn bằng một vector ngắn hơn nhưng vẫn giữ
#        tương đối thông tin ngữ nghĩa/chủ đề quan trọng.
#
#   3. Normalize vector sau SVD.
#      - Chuẩn hóa vector để độ dài vector không ảnh hưởng quá mạnh đến clustering.
#      - Sau normalize, KMeans tập trung hơn vào hướng/ngữ nghĩa của vector.
#
#   4. Dùng KMeans để gom document thành các cụm.
#      - Mỗi document được gán vào một cluster_id.
#      - Các document cùng cluster thường có nội dung/chủ đề tương tự nhau.
#
#   5. Tạo mapping giữa document và cluster.
#      - doc_to_cluster: biết mỗi document thuộc cluster nào.
#      - cluster_to_docs: biết mỗi cluster chứa những document nào.
#
#   6. Khi có query mới:
#      - Chạy BM25 hoặc VSM để lấy top retrieve_k document ban đầu.
#      - Ví dụ retrieve_k=100 nghĩa là lấy 100 tài liệu đầu tiên từ BM25/VSM.
#
#   7. Xác định cluster quan trọng với query.
#      - Lấy top_cluster_docs document đầu tiên trong ranking ban đầu.
#      - Ví dụ top_cluster_docs=20 nghĩa là xét 20 tài liệu top đầu.
#      - Đếm xem các tài liệu top đầu rơi vào cluster nào nhiều nhất.
#      - Cluster nào xuất hiện nhiều trong top đầu thì được xem là quan trọng hơn.
#
#   8. Tính cluster_score.
#      - cluster_score nằm trong khoảng [0, 1].
#      - Cluster xuất hiện nhiều nhất được điểm 1.0.
#      - Các cluster khác được điểm = số lần xuất hiện / số lần xuất hiện lớn nhất.
#
#   9. Kết hợp retrieval_score và cluster_score.
#      - retrieval_score là điểm BM25/VSM ban đầu.
#      - cluster_score là điểm dựa trên cụm.
#      - retrieval_score_norm = retrieval_score / max_score
#      - Công thức:
#
#            final_score = alpha * retrieval_score_norm
#                          + (1 - alpha) * cluster_score
#
#      - alpha=0.85 nghĩa là:
#          + 85% điểm đến từ BM25/VSM ban đầu.
#          + 15% điểm đến từ cluster.
#      - Vì cluster chỉ chiếm 15%, nó chỉ boost nhẹ chứ không phá ranking gốc quá mạnh.
#
#   10. Sort lại tài liệu theo final_score.
#       - Đây là bước reranking.
#       - Kết quả cuối cùng là danh sách doc_id đã được sắp xếp lại.
#
#   11. Chạy reranking cho toàn bộ query.
#       - Tạo kết quả mới cho VSM + Cluster Reranking.
#       - Tạo kết quả mới cho BM25 + Cluster Reranking.
#
#   12. Đánh giá lại bằng evaluate_model.
#       - So sánh baseline với mô hình sau reranking.
#       - Các chỉ số được so sánh gồm MAP, P@20, Recall@20.
#
# Tóm tắt dễ nhớ:
#   BM25/VSM lấy ranking ban đầu
#       -> xem top documents thuộc cluster nào nhiều
#       -> cluster phổ biến được xem là chủ đề liên quan
#       -> document trong cluster liên quan được cộng điểm nhẹ
#       -> sort lại ranking
#       -> đánh giá xem có tốt hơn baseline không
# ============================================================

# Import thuật toán KMeans để gom các document thành nhiều cụm/chủ đề.
from sklearn.cluster import KMeans

# Import TruncatedSVD để giảm chiều ma trận TF-IDF.
# Đây là kỹ thuật gần giống LSA/LSI trong Information Retrieval.
from sklearn.decomposition import TruncatedSVD

# Import normalize để chuẩn hóa vector document về cùng độ dài.
from sklearn.preprocessing import normalize

# defaultdict giúp tạo dictionary mà mỗi key mặc định là một list rỗng.
# Counter giúp đếm số lần xuất hiện của từng cluster.
from collections import defaultdict, Counter

# pandas dùng để tạo bảng so sánh kết quả cuối cùng.
import pandas as pd

# numpy thường dùng cho tính toán ma trận/vector.
# Trong đoạn này numpy chưa dùng trực tiếp, nhưng có thể đã được giữ để dùng mở rộng.
import numpy as np

# ============================================================
# 1. Reduce TF-IDF dimensions
# ============================================================

# In thông báo để biết chương trình đang bắt đầu bước giảm chiều TF-IDF.
print("Reducing TF-IDF dimensions...")

# Khởi tạo mô hình TruncatedSVD.
# n_components=100 nghĩa là giảm vector TF-IDF về 100 chiều.
# random_state=42 giúp kết quả ổn định, chạy lại nhiều lần vẫn giống nhau.
svd = TruncatedSVD(
    n_components=100,
    random_state=42
)

# fit_transform gồm 2 việc:
#   - fit: học các latent dimensions quan trọng từ tfidf_matrix.
#   - transform: biến mỗi document từ vector TF-IDF gốc sang vector 100 chiều.
# tfidf_matrix là ma trận document-term đã được tạo ở các phần trước.
reduced_docs = svd.fit_transform(tfidf_matrix)

# Chuẩn hóa vector sau khi giảm chiều.
# Sau bước này, mỗi document vector có độ dài xấp xỉ 1.
# Việc này giúp clustering ổn định hơn vì KMeans không bị ảnh hưởng quá nhiều bởi độ lớn vector.
reduced_docs = normalize(reduced_docs)

# ============================================================
# 2. KMeans clustering
# ============================================================

# Số lượng cluster muốn chia toàn bộ document.
# Ở đây chia thành 200 cụm.
# Nếu corpus nhỏ quá, 200 có thể hơi nhiều; nếu corpus lớn thì hợp lý hơn.
N_CLUSTERS = 200

# In thông báo số cluster đang được xây dựng.
print(f"Building {N_CLUSTERS} clusters...")

# Khởi tạo KMeans.
# n_clusters=N_CLUSTERS: số cụm cần tạo.
# random_state=42: cố định random để tái lập kết quả.
# n_init=10: chạy KMeans 10 lần với các centroid khởi tạo khác nhau,
#            sau đó chọn kết quả tốt nhất theo inertia.
kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=42,
    n_init=10
)

# fit_predict gồm 2 việc:
#   - fit: học centroid của các cluster từ reduced_docs.
#   - predict: gán mỗi document vào một cluster.
# cluster_labels là mảng có độ dài bằng số document.
# cluster_labels[i] là cluster_id của document ở vị trí i.
cluster_labels = kmeans.fit_predict(reduced_docs)

# ============================================================
# 3. Build cluster mappings
# ============================================================

# Dictionary ánh xạ từ doc_id sang cluster_id.
# Ví dụ: doc_to_cluster[15] = 3 nghĩa là document 15 thuộc cluster 3.
doc_to_cluster = {}

# Dictionary ánh xạ từ cluster_id sang danh sách doc_id thuộc cluster đó.
# defaultdict(list) giúp khi gọi cluster_to_docs[cluster_id] chưa tồn tại,
# nó tự tạo list rỗng.
cluster_to_docs = defaultdict(list)

# Duyệt qua toàn bộ nhãn cluster của từng document.
# idx là vị trí document trong ma trận, bắt đầu từ 0.
# cluster_id là cụm mà document đó được KMeans gán vào.
for idx, cluster_id in enumerate(cluster_labels):

    # Trong code này doc_id được quy ước bắt đầu từ 1.
    # Vì idx bắt đầu từ 0 nên doc_id = idx + 1.
    # Lưu ý: cách này chỉ đúng nếu doc_id thật sự trùng với thứ tự dòng trong tfidf_matrix.
    doc_id = idx + 1

    # Lưu document này thuộc cluster nào.
    doc_to_cluster[doc_id] = cluster_id

    # Thêm document này vào danh sách document của cluster tương ứng.
    cluster_to_docs[cluster_id].append(doc_id)

# In tiêu đề thống kê cluster.
print("\nCluster statistics:")

# In số document trong 5 cluster đầu tiên để kiểm tra nhanh.
# min(5, N_CLUSTERS) tránh lỗi nếu số cluster nhỏ hơn 5.
for cid in range(min(5, N_CLUSTERS)):
    print(f"Cluster {cid}: {len(cluster_to_docs[cid])} docs")

# ============================================================
# 4. Cluster-aware reranking
# ============================================================

# Hàm rerank kết quả truy xuất dựa trên thông tin cluster.
def cluster_rerank(
    query,
    model="bm25",
    retrieve_k=1400,
    top_cluster_docs=20,
    alpha=0.85
):
    """
    Rerank danh sách document trả về từ BM25 hoặc VSM bằng cluster score.

    Parameters
    ----------
    query : str
        Câu truy vấn cần tìm kiếm.

    model : str
        Chọn mô hình retrieval ban đầu.
        Có 2 giá trị hợp lệ:
            - "bm25": dùng BM25.
            - "vsm": dùng Vector Space Model / TF-IDF cosine similarity.

    retrieve_k : int
        Số document lấy từ ranking ban đầu.
        Ví dụ retrieve_k=100 nghĩa là lấy top 100 document từ BM25/VSM.

    top_cluster_docs : int
        Số document top đầu dùng để xác định cluster nào quan trọng.
        Ví dụ top_cluster_docs=20 nghĩa là chỉ nhìn 20 document đầu tiên.

    alpha : float
        Trọng số của retrieval score ban đầu.
        Công thức:
            final_score = alpha * retrieval_score_norm
                          + (1 - alpha) * cluster_score

        alpha càng cao:
            - Ranking càng giống BM25/VSM gốc.
            - Cluster ảnh hưởng ít hơn.

        alpha càng thấp:
            - Cluster ảnh hưởng mạnh hơn.
            - Có nguy cơ làm xáo trộn ranking gốc nhiều hơn.

    Returns
    -------
    list[int]
        Danh sách doc_id sau khi rerank, đã sắp xếp theo final_score giảm dần.
    """

    # --------------------------------------------------------
    # Step 1: Initial retrieval
    # --------------------------------------------------------

    # Nếu chọn BM25 thì gọi hàm search_bm25 đã xây dựng từ phần trước.
    if model == "bm25":

        # search_bm25 trả về danh sách tuple dạng:
        #   [(doc_id, score), (doc_id, score), ...]
        # k=retrieve_k nghĩa là lấy top retrieve_k document.
        initial_results = search_bm25(
            query,
            bm25_model,
            k=retrieve_k
        )

    # Nếu chọn VSM thì gọi hàm search_vsm đã xây dựng từ phần trước.
    elif model == "vsm":

        # search_vsm cũng trả về danh sách tuple dạng:
        #   [(doc_id, score), (doc_id, score), ...]
        # tfidf_matrix: ma trận TF-IDF document-term.
        # vsm_index, vocab_index: các cấu trúc chỉ mục phục vụ truy xuất VSM.
        initial_results = search_vsm(
            query,
            tfidf_matrix,
            vsm_index,
            vocab_index,
            k=retrieve_k
        )

    # Nếu model không phải bm25/vsm thì báo lỗi rõ ràng.
    else:
        raise ValueError("model must be 'bm25' or 'vsm'")

    # --------------------------------------------------------
    # Step 2: Original score map
    # --------------------------------------------------------

    # Tạo dictionary để tra cứu nhanh retrieval score ban đầu theo doc_id.
    # Ví dụ: score_map[10] = 3.25 nghĩa là document 10 có score ban đầu 3.25.
    score_map = {
        doc_id: score
        for doc_id, score in initial_results
    }

    # --------------------------------------------------------
    # Step 3: Compute cluster importance
    # --------------------------------------------------------

    # Lấy doc_id của top_cluster_docs document đầu tiên trong ranking ban đầu.
    # Đây là các document mà BM25/VSM tin là liên quan nhất.
    top_docs = [
        doc_id
        for doc_id, _ in initial_results[:top_cluster_docs]
    ]

    # Đếm số document top đầu rơi vào từng cluster.
    # Ví dụ kết quả có thể là:
    #   Counter({5: 8, 12: 4, 3: 2})
    # nghĩa là trong top_docs có 8 document thuộc cluster 5,
    # 4 document thuộc cluster 12, 2 document thuộc cluster 3.
    cluster_counts = Counter(
        doc_to_cluster[d]
        for d in top_docs
    )

    # Lấy số lần xuất hiện lớn nhất của một cluster.
    # Dùng giá trị này để normalize cluster score về khoảng [0, 1].
    max_count = max(cluster_counts.values())

    # Normalize cluster scores to [0,1].
    # Cluster xuất hiện nhiều nhất sẽ có score = 1.0.
    # Cluster khác sẽ có score = count / max_count.
    # Cluster không xuất hiện trong top_docs sẽ không nằm trong dictionary này.
    cluster_scores = {
        cluster_id: count / max_count
        for cluster_id, count in cluster_counts.items()
    }

    # --------------------------------------------------------
    # Step 4: Hybrid reranking
    # --------------------------------------------------------

    # Danh sách lưu kết quả sau khi tính final_score.
    # Mỗi phần tử sẽ có dạng: (doc_id, final_score).
    reranked = []

    # Lấy retrieval score lớn nhất trong kết quả ban đầu.
    # Dùng để normalize retrieval_score về khoảng tương đối [0, 1].
    max_retrieval_score = max(score_map.values())

    # Duyệt qua từng document trong ranking ban đầu.
    for doc_id, retrieval_score in initial_results:

        # Lấy cluster_id của document hiện tại.
        cluster_id = doc_to_cluster[doc_id]

        # Lấy cluster_score của cluster chứa document này.
        # Nếu cluster của document không xuất hiện trong top_docs,
        # cluster_score được gán là 0.0.
        cluster_score = cluster_scores.get(cluster_id, 0.0)

        # Normalize retrieval score.
        # Ví dụ document có score bằng max score thì retrieval_score_norm = 1.0.
        # Document có score bằng một nửa max score thì retrieval_score_norm = 0.5.
        retrieval_score_norm = retrieval_score / max_retrieval_score

        # Tính điểm cuối cùng bằng cách kết hợp:
        #   - retrieval_score_norm: điểm truy xuất ban đầu đã chuẩn hóa.
        #   - cluster_score: điểm cụm/chủ đề.
        # alpha=0.85 nghĩa là giữ 85% ảnh hưởng từ retrieval score,
        # còn cluster chỉ boost nhẹ 15%.
        final_score = (
            alpha * retrieval_score_norm
            +
            (1 - alpha) * cluster_score
        )

        # Lưu document cùng final_score để lát nữa sort lại.
        reranked.append(
            (doc_id, final_score)
        )

    # Sort danh sách theo final_score giảm dần.
    # Document có final_score cao nhất sẽ đứng đầu.
    reranked = sorted(
        reranked,
        key=lambda x: x[1],
        reverse=True
    )

    # Chỉ trả về danh sách doc_id, bỏ final_score.
    # evaluate_model thường chỉ cần ranking document, không cần score.
    return [
        doc_id
        for doc_id, _ in reranked
    ]

# ============================================================
# 5. Run reranking for all queries
# ============================================================

# In thông báo bắt đầu reranking cho toàn bộ query.
print("\nRunning cluster-aware reranking...")

# Dictionary lưu kết quả VSM sau cluster reranking.
# Key là qid, value là ranking doc_id sau rerank.
vsm_cluster_results = {}

# Dictionary lưu kết quả BM25 sau cluster reranking.
# Key là qid, value là ranking doc_id sau rerank.
bm25_cluster_results = {}

# Duyệt qua toàn bộ query trong tập queries.
# queries có dạng thường là:
#   {qid: query_text}
for qid, query_text in queries.items():

    # Chạy cluster reranking cho VSM.
    # model="vsm" nghĩa là ranking ban đầu lấy từ search_vsm.
    # retrieve_k=1400 nghĩa là rerank trong phạm vi top 1400 document.
    # top_cluster_docs=20 nghĩa là dùng top 20 document để xác định cluster quan trọng.
    # alpha=0.85 nghĩa là retrieval score chiếm 85%, cluster score chiếm 15%.
    vsm_cluster_results[qid] = cluster_rerank(
        query=query_text,
        model="vsm",
        retrieve_k=1400,
        top_cluster_docs=20,
        alpha=0.85
    )

    # Chạy cluster reranking cho BM25 với cùng tham số.
    bm25_cluster_results[qid] = cluster_rerank(
        query=query_text,
        model="bm25",
        retrieve_k=1400,
        top_cluster_docs=20,
        alpha=0.85
    )

# ============================================================
# 6. Evaluate
# ============================================================

# In thông báo bắt đầu đánh giá mô hình sau reranking.
print("\nEvaluating reranked models...")

# Đánh giá kết quả VSM + Cluster Reranking.
# vsm_cluster_results: ranking sau rerank cho từng query.
# query_rels: ground-truth relevance judgments.
# k=20: tính các metric tại top 20 nếu evaluate_model có P@20/Recall@20.
vsm_cluster_metrics = evaluate_model(
    vsm_cluster_results,
    query_rels,
    k=20
)

# Đánh giá kết quả BM25 + Cluster Reranking.
bm25_cluster_metrics = evaluate_model(
    bm25_cluster_results,
    query_rels,
    k=20
)

# ============================================================
# 7. Compare with baselines
# ============================================================

# Tạo DataFrame để so sánh baseline với phiên bản có cluster reranking.
# Mỗi dictionary là một dòng trong bảng.
comparison_df = pd.DataFrame([

    # Dòng kết quả VSM gốc, chưa rerank.
    {
        "Model": "VSM Baseline",
        "MAP": vsm_metrics["MAP"],
        "P@20": vsm_metrics["P@20"],
        "R@20": vsm_metrics["Recall@20"]
    },

    # Dòng kết quả VSM sau khi thêm cluster reranking.
    {
        "Model": "VSM + Cluster Reranking",
        "MAP": vsm_cluster_metrics["MAP"],
        "P@20": vsm_cluster_metrics["P@20"],
        "R@20": vsm_cluster_metrics["Recall@20"]
    },

    # Dòng kết quả BM25 gốc, chưa rerank.
    {
        "Model": "BM25 Baseline",
        "MAP": bm25_metrics["MAP"],
        "P@20": bm25_metrics["P@20"],
        "R@20": bm25_metrics["Recall@20"]
    },

    # Dòng kết quả BM25 sau khi thêm cluster reranking.
    {
        "Model": "BM25 + Cluster Reranking",
        "MAP": bm25_cluster_metrics["MAP"],
        "P@20": bm25_cluster_metrics["P@20"],
        "R@20": bm25_cluster_metrics["Recall@20"]
    }

])

# Hiển thị bảng so sánh trong notebook.
# Nếu chạy trong Jupyter/Colab thì display sẽ render thành bảng đẹp.
display(comparison_df)

# ============================================================
# 8. Notes
# ============================================================

# In một số ghi chú giúp giải thích kết quả và ý tưởng của phương pháp.
print("\nNotes:")

# Cluster score chỉ đóng vai trò boost nhẹ, không thay thế hoàn toàn điểm BM25/VSM.
print("- Cluster score only boosts retrieval score lightly.")

# Ranking gốc vẫn là tín hiệu chính vì alpha=0.85.
print("- Retrieval ranking is preserved and not overridden.")

# MAP 
print("- MAP uses top-k retrieved documents.") 

# P@20 và Recall@20 chỉ xét 20 document đầu tiên sau reranking.
print("- P@20 and Recall@20 use top-20 results.")

# Mục tiêu của cluster reranking là cải thiện độ phủ chủ đề mà vẫn giữ chất lượng ranking.
print("- Cluster reranking aims to improve topical coverage while preserving ranking quality.")


Reducing TF-IDF dimensions...
Building 200 clusters...

Cluster statistics:
Cluster 0: 10 docs
Cluster 1: 10 docs
Cluster 2: 6 docs
Cluster 3: 8 docs
Cluster 4: 8 docs

Running cluster-aware reranking...

Evaluating reranked models...


,Model,MAP,P@20,R@20
0,VSM Baseline,0.2923,0.1573,0.5048
1,VSM + Cluster Reranking,0.3045,0.1660,0.5305
2,BM25 Baseline,0.3118,0.1622,0.5182
3,BM25 + Cluster Reranking,0.3297,0.1720,0.5500



Notes:
- Cluster score only boosts retrieval score lightly.
- Retrieval ranking is preserved and not overridden.
- MAP uses top-k retrieved documents.
- P@20 and Recall@20 use top-20 results.
- Cluster reranking aims to improve topical coverage while preserving ranking quality.


## Benchmark 1 vài cách:
Query specific:

1. Query q đi vào BM25/VSM
2. Lấy top100 documents ban đầu
3. Chỉ cluster 100 documents này, không cluster toàn corpus
4. Lấy top20 documents đầu để vote cluster
5. Cluster nào có nhiều top docs hơn → cluster đó có khả năng relevant hơn
6. Mỗi doc trong top100 nhận thêm điểm cluster theo cluster của nó
7. Sort lại theo FinalScore

In [76]:
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from collections import Counter, defaultdict
import numpy as np
import pandas as pd


def get_initial_results(query, model="bm25", k=1400):
    if model == "bm25":
        return search_bm25(query, bm25_model, k=k)
    elif model == "vsm":
        return search_vsm(query, tfidf_matrix, vsm_index, vocab_index, k=k)
    else:
        raise ValueError("model must be 'bm25' or 'vsm'")


def normalize_score_map(results):
    scores = {doc_id: score for doc_id, score in results}
    max_score = max(scores.values()) if scores else 1.0
    if max_score == 0:
        max_score = 1.0
    return {d: s / max_score for d, s in scores.items()}


def static_cluster_rerank(query, model="bm25", retrieve_k=1400, top_cluster_docs=20, alpha=0.85):
    results = get_initial_results(query, model, retrieve_k)
    score_map = normalize_score_map(results)

    top_docs = [doc_id for doc_id, _ in results[:top_cluster_docs]]
    cluster_counts = Counter(doc_to_cluster[d] for d in top_docs)
    max_count = max(cluster_counts.values()) if cluster_counts else 1
    cluster_scores = {cid: count / max_count for cid, count in cluster_counts.items()}

    reranked = []
    for doc_id, _ in results:
        cid = doc_to_cluster[doc_id]
        cluster_score = cluster_scores.get(cid, 0.0)
        final_score = alpha * score_map.get(doc_id, 0.0) + (1 - alpha) * cluster_score
        reranked.append((doc_id, final_score))

    reranked = sorted(reranked, key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in reranked]


def query_specific_cluster_rerank(
    query, model="bm25", retrieve_k=1400,
    n_local_clusters=8, top_cluster_docs=20, alpha=0.85
):
    results = get_initial_results(query, model, retrieve_k)
    doc_list = [doc_id for doc_id, _ in results]
    score_map = normalize_score_map(results)

    if len(doc_list) < n_local_clusters:
        return doc_list

    rows = [doc_id - 1 for doc_id in doc_list]
    local_vectors = reduced_docs[rows]

    local_kmeans = KMeans(n_clusters=n_local_clusters, random_state=42, n_init=10)
    local_labels = local_kmeans.fit_predict(local_vectors)

    local_doc_to_cluster = {doc_id: int(local_labels[i]) for i, doc_id in enumerate(doc_list)}

    top_docs = doc_list[:top_cluster_docs]
    cluster_counts = Counter(local_doc_to_cluster[d] for d in top_docs)
    max_count = max(cluster_counts.values()) if cluster_counts else 1
    cluster_scores = {cid: count / max_count for cid, count in cluster_counts.items()}

    reranked = []
    for doc_id in doc_list:
        cid = local_doc_to_cluster[doc_id]
        cluster_score = cluster_scores.get(cid, 0.0)
        final_score = alpha * score_map.get(doc_id, 0.0) + (1 - alpha) * cluster_score
        reranked.append((doc_id, final_score))

    reranked = sorted(reranked, key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in reranked]


def light_expand_rerank(
    query, model="bm25", retrieve_k=1400,
    seed_k=3, expand_per_seed=3, final_k=1400, alpha=0.9
):
    results = get_initial_results(query, model, retrieve_k)
    base_docs = [doc_id for doc_id, _ in results]
    score_map = normalize_score_map(results)

    candidates = set(base_docs)
    seed_docs = base_docs[:seed_k]

    for seed_doc in seed_docs:
        seed_vec = reduced_docs[seed_doc - 1]
        cid = doc_to_cluster[seed_doc]
        same_cluster_docs = cluster_to_docs[cid]

        sims = []
        for d in same_cluster_docs:
            if d in candidates:
                continue
            sim = float(np.dot(seed_vec, reduced_docs[d - 1]))
            sims.append((d, sim))

        sims = sorted(sims, key=lambda x: x[1], reverse=True)
        candidates.update([d for d, _ in sims[:expand_per_seed]])

    full_results = get_initial_results(query, model, k=len(documents))
    full_score_map = normalize_score_map(full_results)

    seed_clusters = Counter(doc_to_cluster[d] for d in seed_docs)
    max_count = max(seed_clusters.values()) if seed_clusters else 1
    cluster_scores = {cid: count / max_count for cid, count in seed_clusters.items()}

    reranked = []
    for doc_id in candidates:
        cid = doc_to_cluster[doc_id]
        cluster_score = cluster_scores.get(cid, 0.0)
        final_score = alpha * full_score_map.get(doc_id, 0.0) + (1 - alpha) * cluster_score
        reranked.append((doc_id, final_score))

    reranked = sorted(reranked, key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in reranked[:final_k]]


def run_strategy(strategy_func, model="bm25", **kwargs):
    results = {}
    for qid, query_text in queries.items():
        results[qid] = strategy_func(query_text, model=model, **kwargs)
    return results


experiments = []

configs = [
    {
        "name": "Static Cluster Rerank",
        "func": static_cluster_rerank,
        "kwargs": {"retrieve_k": 1400, "top_cluster_docs": 20, "alpha": 0.85}
    },
    {
        "name": "Query-specific Cluster Rerank",
        "func": query_specific_cluster_rerank,
        "kwargs": {"retrieve_k": 1400, "n_local_clusters": 8, "top_cluster_docs": 20, "alpha": 0.85}
    },
    {
        "name": "Light Cluster Expansion",
        "func": light_expand_rerank,
        "kwargs": {"retrieve_k": 1400, "seed_k": 3, "expand_per_seed": 3, "final_k": 1400, "alpha": 0.9}
    }
]

for model_name in ["vsm", "bm25"]:
    for cfg in configs:
        print(f"Running {model_name.upper()} - {cfg['name']}...")
        result_dict = run_strategy(cfg["func"], model=model_name, **cfg["kwargs"])
        metrics = evaluate_model(result_dict, query_rels, k=20)
        experiments.append({
            "Model": model_name.upper(),
            "Strategy": cfg["name"],
            "MAP": metrics["MAP"],
            "P@20": metrics["P@20"],
            "R@20": metrics["Recall@20"]
        })

experiments.insert(0, {
    "Model": "VSM", "Strategy": "Baseline",
    "MAP": vsm_metrics["MAP"], "P@20": vsm_metrics["P@20"], "R@20": vsm_metrics["Recall@20"]
})
experiments.insert(4, {
    "Model": "BM25", "Strategy": "Baseline",
    "MAP": bm25_metrics["MAP"], "P@20": bm25_metrics["P@20"], "R@20": bm25_metrics["Recall@20"]
})

experiment_df = pd.DataFrame(experiments)
display(experiment_df.sort_values(["Model", "MAP"], ascending=[True, False]))

print("\nBest by MAP:")
display(experiment_df.sort_values("MAP", ascending=False).head(5))


Running VSM - Static Cluster Rerank...
Running VSM - Query-specific Cluster Rerank...
Running VSM - Light Cluster Expansion...
Running BM25 - Static Cluster Rerank...
Running BM25 - Query-specific Cluster Rerank...
Running BM25 - Light Cluster Expansion...


,Model,Strategy,MAP,P@20,R@20
5,BM25,Static Cluster Rerank,0.3297,0.1720,0.5500
7,BM25,Light Cluster Expansion,0.3296,0.1720,0.5503
6,BM25,Query-specific Cluster Rerank,0.3173,0.1651,0.5261
4,BM25,Baseline,0.3118,0.1622,0.5182
3,VSM,Light Cluster Expansion,0.3067,0.1673,0.5369
1,VSM,Static Cluster Rerank,0.3045,0.1660,0.5305
2,VSM,Query-specific Cluster Rerank,0.2964,0.1629,0.5194
0,VSM,Baseline,0.2923,0.1573,0.5048



Best by MAP:


,Model,Strategy,MAP,P@20,R@20
5,BM25,Static Cluster Rerank,0.3297,0.1720,0.5500
7,BM25,Light Cluster Expansion,0.3296,0.1720,0.5503
6,BM25,Query-specific Cluster Rerank,0.3173,0.1651,0.5261
4,BM25,Baseline,0.3118,0.1622,0.5182
3,VSM,Light Cluster Expansion,0.3067,0.1673,0.5369


In [ ]:
# Kiểm thử đối chiếu tự động giữa BM25 cũ (Legacy) và BM25 mới (Chỉ mục đảo ngược)
print("Bắt đầu chạy kiểm thử so sánh độ chính xác...")
legacy_model = BM25_Legacy(processed_docs)
new_model = BM25(processed_docs)

mismatch_count = 0
for qid, qtext in queries.items():
    tokens = process_document(qtext)
    legacy_res = legacy_model.get_scores(tokens, k=20)
    new_res = new_model.get_scores(tokens, k=20)
    
    if len(legacy_res) != len(new_res):
        mismatch_count += 1
        continue
    
    for (d1, s1), (d2, s2) in zip(legacy_res, new_res):
        if d1 != d2 or not math.isclose(s1, s2, rel_tol=1e-9):
            mismatch_count += 1
            break

if mismatch_count == 0:
    print("XÁC NHẬN: Kết quả của mô hình BM25 mới dùng chỉ mục đảo ngược TRÙNG KHỚP 100% so với mô hình cũ!")
else:
    print(f"CẢNH BÁO: Phát hiện {mismatch_count} điểm không khớp trong kết quả tìm kiếm!")


## 8. Whoosh Baseline Comparison

Phần này dùng **Whoosh BM25F** như một baseline thư viện để so sánh với các mô hình đã cài đặt thủ công trên cùng bộ Cranfield.

Section này được viết để **chạy độc lập** và dùng lại **cùng preprocessing pipeline** của notebook: lowercase, mở rộng viết tắt, chuyển số thành chữ, tokenize, stopword removal và Snowball stemming. Documents và queries đều được preprocess trước khi đưa vào Whoosh để so sánh công bằng hơn với VSM/BM25 thủ công.


In [1]:
# Nếu môi trường chưa có Whoosh, chạy cell này trước.
try:
    import whoosh
    print("Whoosh đã sẵn sàng.")
except ImportError:
    %pip install whoosh


Whoosh đã sẵn sàng.


In [3]:
import csv
import os
import re
import shutil
from collections import defaultdict
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import word_tokenize
from num2words import num2words
from whoosh import scoring
from whoosh.analysis import KeywordAnalyzer
from whoosh.fields import ID, TEXT, Schema
from whoosh.index import create_in, open_dir
from whoosh.qparser import OrGroup, QueryParser

for pkg in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(pkg, quiet=True)

CRANFIELD_DIR = Path("Cranfield")
QUERY_FILE = Path("TEST") / "query.txt"
RELEVANCE_DIR = Path("TEST") / "RES"
WHOOSH_INDEX_DIR = Path("whoosh_cranfield_index")
TOP_K = 20

ABBREVIATIONS = {
    r"\bfig\.?\b": "figure",
    r"\bref\.?\b": "reference",
    r"\bapprox\.?\b": "approximately",
    r"\beq\.?\b": "equation",
    r"\bsq\.?\b": "square",
    r"\bno\.?\b": "number",
    r"\be\.g\.?\b": "for example",
    r"\bi\.e\.?\b": "that is",
    r"\bsec\.?\b": "section",
}
CUSTOM_STOPWORDS = {"ii", "iii", "iv", "vi", "vii", "viii", "ix", "xi", "xii"}
STOP_WORDS = set(stopwords.words("english")) | CUSTOM_STOPWORDS
STEMMER = SnowballStemmer("english")


def replace_number(match):
    try:
        return num2words(float(match.group())).replace("-", " ").replace(",", "")
    except Exception:
        return match.group()


def process_document(document, stop_words=None, stemmer=None):
    if stop_words is None:
        stop_words = STOP_WORDS
    if stemmer is None:
        stemmer = STEMMER

    text = document.lower().replace("-", " ")
    for pattern, replacement in ABBREVIATIONS.items():
        text = re.sub(pattern, replacement, text)

    text = re.sub(r"\b\d+(?:\.\d+)?\b", replace_number, text)
    text = re.sub(r"[^a-z0-9\s]", "", text)

    tokens = word_tokenize(text)
    tokens = [
        token for token in tokens
        if not (len(token) <= 2 and token.isalpha()) and token not in stop_words
    ]
    return [stemmer.stem(token) for token in tokens]


def preprocess_for_whoosh(text):
    return " ".join(process_document(text))


def load_documents(folder):
    documents = {}
    for file_name in sorted(os.listdir(folder)):
        if file_name.endswith(".txt"):
            doc_id = int(file_name.split(".")[0])
            file_path = Path(folder) / file_name
            documents[doc_id] = file_path.read_text(encoding="utf-8")
    return documents


def load_queries(query_file):
    queries = {}
    with open(query_file, encoding="utf-8") as file:
        for row in csv.reader(file, delimiter="\t"):
            if len(row) >= 2:
                queries[int(row[0])] = row[1]
    return queries


def load_relevance(result_path):
    relevance = defaultdict(list)
    for file_name in os.listdir(result_path):
        qid = int(file_name.split(".")[0])
        file_path = Path(result_path) / file_name
        df = pd.read_csv(
            file_path,
            sep=r"\s+",
            header=None,
            names=["QueryID", "DocID", "Rating"],
            engine="python",
        )
        df = df.dropna(subset=["DocID", "Rating"])
        relevance[qid] = [int(row.DocID) for row in df.itertuples() if int(row.Rating) != -1]
    return relevance


def average_precision(retrieved, relevant):
    relevant_set = set(relevant)
    if not relevant_set:
        return 0.0

    score = 0.0
    hits = 0
    for rank, doc_id in enumerate(retrieved, 1):
        if doc_id in relevant_set:
            hits += 1
            score += hits / rank
    return score / len(relevant_set)


def evaluate_model(results, query_rels, k=20):
    aps, precisions, recalls = [], [], []
    for qid, retrieved in results.items():
        if qid not in query_rels:
            continue

        relevant = set(query_rels[qid])
        if not relevant:
            continue

        top_k = retrieved[:k]
        hits = sum(1 for doc_id in top_k if doc_id in relevant)
        precisions.append(hits / k)
        recalls.append(hits / len(relevant))
        aps.append(average_precision(retrieved, query_rels[qid]))

    return {
        "MAP": round(float(np.mean(aps)), 4) if aps else 0.0,
        f"P@{k}": round(float(np.mean(precisions)), 4) if precisions else 0.0,
        f"Recall@{k}": round(float(np.mean(recalls)), 4) if recalls else 0.0,
    }


def build_whoosh_index(documents, index_dir=WHOOSH_INDEX_DIR):
    if index_dir.exists():
        shutil.rmtree(index_dir)
    index_dir.mkdir(parents=True, exist_ok=True)

    schema = Schema(
        doc_id=ID(stored=True, unique=True),
        content=TEXT(analyzer=KeywordAnalyzer(), stored=False),
    )
    index = create_in(index_dir, schema)

    writer = index.writer()
    for doc_id, text in sorted(documents.items()):
        writer.add_document(doc_id=str(doc_id), content=preprocess_for_whoosh(text))
    writer.commit()
    return open_dir(index_dir)


def search_whoosh(query_text, index, k):
    parser = QueryParser("content", schema=index.schema, group=OrGroup)
    parsed_query = parser.parse(preprocess_for_whoosh(query_text))

    with index.searcher(weighting=scoring.BM25F()) as searcher:
        hits = searcher.search(parsed_query, limit=k)
        return [int(hit["doc_id"]) for hit in hits]


whoosh_documents = load_documents(CRANFIELD_DIR)
whoosh_queries = load_queries(QUERY_FILE)
whoosh_query_rels = load_relevance(RELEVANCE_DIR)

print(f"Documents: {len(whoosh_documents)}")
print(f"Queries: {len(whoosh_queries)}")
print(f"Relevance files: {len(whoosh_query_rels)}")
print("Building Whoosh BM25F index with the same preprocessing pipeline...")

whoosh_index = build_whoosh_index(whoosh_documents)
whoosh_results = {
    qid: search_whoosh(query_text, whoosh_index, k=len(whoosh_documents))
    for qid, query_text in whoosh_queries.items()
}

whoosh_metrics = evaluate_model(whoosh_results, whoosh_query_rels, k=TOP_K)
whoosh_metrics


Documents: 1400
Queries: 225
Relevance files: 225
Building Whoosh BM25F index with the same preprocessing pipeline...


{'MAP': 0.303, 'P@20': 0.1607, 'Recall@20': 0.5123}

In [4]:
comparison_with_whoosh = pd.DataFrame([
    {"Model": "VSM Baseline", "MAP": 0.2923, "P@20": 0.1573, "R@20": 0.5048},
    {"Model": "VSM + Cluster Reranking", "MAP": 0.3045, "P@20": 0.1660, "R@20": 0.5305},
    {"Model": "BM25 Baseline", "MAP": 0.3118, "P@20": 0.1622, "R@20": 0.5182},
    {"Model": "BM25 + Cluster Reranking", "MAP": 0.3297, "P@20": 0.1720, "R@20": 0.5500},
    {
        "Model": "Whoosh BM25F Baseline",
        "MAP": whoosh_metrics["MAP"],
        "P@20": whoosh_metrics["P@20"],
        "R@20": whoosh_metrics["Recall@20"],
    },
])

display(comparison_with_whoosh)

print("Best by MAP:")
display(comparison_with_whoosh.sort_values("MAP", ascending=False).head())


,Model,MAP,P@20,R@20
0,VSM Baseline,0.2923,0.1573,0.5048
1,VSM + Cluster Reranking,0.3045,0.1660,0.5305
2,BM25 Baseline,0.3118,0.1622,0.5182
3,BM25 + Cluster Reranking,0.3297,0.1720,0.5500
4,Whoosh BM25F Baseline,0.3030,0.1607,0.5123


Best by MAP:


,Model,MAP,P@20,R@20
3,BM25 + Cluster Reranking,0.3297,0.1720,0.5500
2,BM25 Baseline,0.3118,0.1622,0.5182
1,VSM + Cluster Reranking,0.3045,0.1660,0.5305
4,Whoosh BM25F Baseline,0.3030,0.1607,0.5123
0,VSM Baseline,0.2923,0.1573,0.5048
